# CasCrop: Publication Experiments
**All experiments for Nature Communications submission.**

| Experiment | Purpose | Time (T4) |
|---|---|---|
| Main ablation (5 models x N seeds) | Table 2 | ~4-6 hrs |
| Graph perturbation | Proves structure matters | ~30 min |
| Edge type ablation | Geo vs commodity | ~30 min |
| Disentanglement probe | Encoder independence | ~1 min |
| Statistical tests + figure | Publication outputs | ~1 min |

In [ ]:
#@title Configuration
QUICK_TEST = True  #@param {type:"boolean"}
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 20 if QUICK_TEST else 200
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 1024 if QUICK_TEST else 512
print(f"{'QUICK' if QUICK_TEST else 'FULL'}: {len(SEEDS)} seeds, {EPOCHS} epochs")

In [ ]:
#@title Setup
import torch, os, sys, json, time, shutil, subprocess
import numpy as np, pandas as pd
from pathlib import Path
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB)')
    if torch.cuda.get_device_properties(0).total_mem < 8e9: BATCH_SIZE = 256
else: print('NO GPU')
if not os.path.exists('CasCrop'): 
    !git clone https://github.com/keshavkrishnan08/CasCrop.git
if os.path.basename(os.getcwd()) != 'CasCrop':
    %cd CasCrop
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn tqdm pyyaml 2>&1 | tail -1
sys.path.insert(0, 'src')
for d in ['checkpoints','results','paper/figures','paper/tables']: os.makedirs(d, exist_ok=True)
SAVE_TO_DRIVE = False; DRIVE_PATH = ''
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'
    os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
    for f in Path(f'{DRIVE_PATH}/checkpoints').glob('*.pt'):
        if not Path(f'checkpoints/{f.name}').exists(): shutil.copy2(f, f'checkpoints/{f.name}')
    SAVE_TO_DRIVE = True; print(f'Drive: {DRIVE_PATH}')
except: print('No Drive')
def backup():
    if not SAVE_TO_DRIVE: return
    for d in ['results','checkpoints','paper/figures','paper/tables']:
        if not os.path.exists(d): continue
        dst = f'{DRIVE_PATH}/{d}'; os.makedirs(dst, exist_ok=True)
        for f in Path(d).glob('*'):
            if f.is_file(): shutil.copy2(f, f'{dst}/{f.name}')
print('OK')

In [ ]:
#@title Load Data (54MB auto-download)
if Path('data/processed/features.parquet').exists() and Path('data/graphs/combined_graph.npz').exists():
    print('Data present.')
else:
    !wget -q --show-progress -O d.tar.gz https://github.com/keshavkrishnan08/CasCrop/releases/download/v0.1-data/cascrop_processed.tar.gz
    !tar xzf d.tar.gz && rm d.tar.gz
features = pd.read_parquet('data/processed/features.parquet')
labels = pd.read_parquet('data/processed/labels.parquet')
with open('data/processed/splits.json') as f: splits = json.load(f)
with open('data/processed/feature_groups.json') as f: groups = json.load(f)
print(f'{len(features):,} samples | {features["fips"].nunique()} counties | {labels["waste"].mean():.1%} waste')

---
## Experiment 1: Main Ablation (Table 2)

In [ ]:
import os
if os.path.exists('results/training_results.json'):
    with open('results/training_results.json') as f: res=json.load(f)
    df = pd.DataFrame(res)
    for m in models:
        d=df[df['model']==m]
        if len(d): print(f'{m:<20} AUC={d["test_auc_roc"].mean():.3f}+/-{d["test_auc_roc"].std():.3f}  F1={d["test_f1"].mean():.3f}')
else:
    print('No results yet. Run training cells first.')


---
## Experiment 2: Graph Perturbation

In [ ]:
%%time
g=np.load('data/graphs/combined_graph.npz'); np.random.seed(42)
np.savez('data/graphs/shuffled.npz', edge_index=np.array([g['edge_index'][0],np.random.permutation(g['edge_index'][1])]), edge_weight=g['edge_weight'])
ss3=' '.join(str(s) for s in SEEDS[:3])
subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --graph data/graphs/shuffled.npz', shell=True, capture_output=True, text=True, timeout=3600)
shutil.copy('results/training_results.json','results/perturbation.json')
with open('results/perturbation.json') as f: pr_data=json.load(f)
pr=pd.DataFrame(pr_data)
# Get original CasCrop AUC from main ablation
if os.path.exists('results/training_results.json'):
    with open('results/training_results.json') as f: main_res=json.load(f)
    main_df=pd.DataFrame(main_res)
    orig_auc=main_df[main_df['model']=='cascrop']['test_auc_roc'].mean()
else:
    orig_auc=0.0
    # Re-run main ablation with --resume to get results
    subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume', shell=True, capture_output=True, text=True, timeout=600)
    if os.path.exists('results/training_results.json'):
        with open('results/training_results.json') as f: main_res=json.load(f)
        main_df=pd.DataFrame(main_res)
        orig_auc=main_df[main_df['model']=='cascrop']['test_auc_roc'].mean()
shuf_auc=pr['test_auc_roc'].mean()
print(f'Original: {orig_auc:.3f}  |  Shuffled: {shuf_auc:.3f}  |  Drop: {orig_auc-shuf_auc:+.3f}')
if torch.cuda.is_available(): torch.cuda.empty_cache()


---
## Experiment 3: Edge Type Ablation (Table 4)

In [ ]:
%%time
from scipy import sparse
ss3=' '.join(str(s) for s in SEEDS[:3])  # redefine in case perturbation was skipped
def sp2npz(mat,path,k=20):
    d=mat.toarray();n=d.shape[0];r,c,v=[],[],[]
    for i in range(n):
        nz=np.where(d[i]>0)[0]
        if len(nz)==0:continue
        top=nz[np.argsort(d[i,nz])[-k:]]
        for j in top:r.append(i);c.append(j);v.append(d[i,j])
    np.savez(path,edge_index=np.array([r,c]),edge_weight=np.array(v))
sp2npz(sparse.load_npz('data/graphs/adjacency_geo.npz'),'data/graphs/geo_only.npz')
comm=(sparse.load_npz('data/graphs/adjacency_commodity_corn.npz')+sparse.load_npz('data/graphs/adjacency_commodity_soybeans.npz')+sparse.load_npz('data/graphs/adjacency_commodity_wheat.npz'))/3
sp2npz(comm,'data/graphs/comm_only.npz')
# Get original CasCrop AUC for comparison
orig_auc = 0.0
subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume', shell=True, capture_output=True, text=True, timeout=600)
if os.path.exists('results/training_results.json'):
    with open('results/training_results.json') as f: _r=json.load(f)
    _d=pd.DataFrame(_r); orig_auc=_d[_d['model']=='cascrop']['test_auc_roc'].mean()
for nm,gp in [('geo_only','data/graphs/geo_only.npz'),('commodity_only','data/graphs/comm_only.npz')]:
    subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --graph {gp}', shell=True, capture_output=True, text=True, timeout=3600)
    with open('results/training_results.json') as f: rd_data=json.load(f)
    rd=pd.DataFrame(rd_data)
    print(f'{nm:20s}: {rd["test_auc_roc"].mean():.3f}+/-{rd["test_auc_roc"].std():.3f}')
    if torch.cuda.is_available():torch.cuda.empty_cache()
print(f'{"combined (CasCrop)":20s}: {orig_auc:.3f}')
backup()


---
## Experiment 4: Disentanglement Probe

In [ ]:
from models.cascrop import CasCrop
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
with open('data/processed/stats.json') as f: stats=json.load(f)
dev='cuda' if torch.cuda.is_available() else 'cpu'
mdl=CasCrop(bio_input_dim=len(groups['biophysical']),econ_input_dim=len(groups['economic']),hist_dim=len(groups['historical']),latent_dim=64,num_heads=4,dropout=0.0).to(dev)
mdl.load_state_dict(torch.load('checkpoints/cascrop_seed42.pt',map_location=dev,weights_only=False)['model_state_dict']);mdl.eval()
tf=features.iloc[splits['test']]
def nm(df,cols):
    X=df[cols].values.astype(np.float32)
    for i,c in enumerate(cols):
        if c in stats:X[:,i]=(X[:,i]-stats[c]['mean'])/max(stats[c]['std'],1e-8)
    return torch.from_numpy(np.nan_to_num(X,0.0))
b={'x_bio':nm(tf,groups['biophysical']).to(dev),'x_econ':nm(tf,groups['economic']).to(dev),'x_hist':nm(tf,groups['historical']).to(dev),'edge_index':torch.stack([torch.arange(len(tf)),torch.arange(len(tf))]).to(dev),'edge_attr':None,'price_shocks':torch.zeros(len(tf),1).to(dev)}
with torch.no_grad():out=mdl(b)
zb,ze=out['z_bio'].cpu().numpy(),out['z_econ'].cpu().numpy()
el=KMeans(5,random_state=42,n_init=10).fit_predict(ze)
zs=StandardScaler().fit_transform(zb);h=len(zb)//2
acc=LogisticRegression(max_iter=1000,random_state=42).fit(zs[:h],el[:h]).score(zs[h:],el[h:])
print(f'Probe accuracy: {acc:.3f}  (pass if < 0.55, random = 0.20)')
del mdl
if torch.cuda.is_available():torch.cuda.empty_cache()

---
## Results: Stats + Figure + Table

In [ ]:
# Restore main results
ss=' '.join(str(s) for s in SEEDS)
subprocess.run(f'python scripts/04_train_all.py --seeds {ss} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume', shell=True, capture_output=True, text=True, timeout=600)
if not os.path.exists('results/training_results.json'):
    print('ERROR: No training results found. Run Experiment 1 first.'); raise SystemExit
with open('results/training_results.json') as f: res=json.load(f)
df=pd.DataFrame(res)

# Statistical tests
from evaluation.statistical_tests import paired_ttest_across_seeds
ca=sorted(df[df['model']=='cascrop']['test_auc_roc'].tolist())
if len(ca) < 2:
    print('Need >= 2 seeds for paired tests. Showing raw results instead.')
    for m in df['model'].unique():
        d=df[df['model']==m]
        print(f'{m:<20} AUC={d["test_auc_roc"].mean():.3f}')
else:
    print(f'{"Comparison":<35} {"DAUC":>7} {"p":>8} {"Sig":>5}')
    print('-'*58)
    for m in ['local_only','local_econ','geo_gat','symmetric_ecmp']:
        ma=sorted(df[df['model']==m]['test_auc_roc'].tolist())
        if len(ma)!=len(ca):continue
        t=paired_ttest_across_seeds(ca,ma)
        sig='***' if t['p_value']<.001 else '**' if t['p_value']<.01 else '*' if t['p_value']<.05 else 'n.s.'
        print(f'CasCrop vs {m:<23} {t["mean_diff"]:>+.4f} {t["p_value"]:>8.4f} {sig:>5}')
c=df[df['model']=='cascrop']['test_auc_roc'].mean()
l=df[df['model']=='local_only']['test_auc_roc'].mean() if len(df[df['model']=='local_only'])>0 else 0
g=df[df['model']=='geo_gat']['test_auc_roc'].mean() if len(df[df['model']=='geo_gat'])>0 else 0
s=df[df['model']=='symmetric_ecmp']['test_auc_roc'].mean() if len(df[df['model']=='symmetric_ecmp'])>0 else 0
print(f'\nH1 Graph>Indep:    +{c-l:.3f} {"CONFIRMED" if c>l else "FAILED"}')
print(f'H2 Econ>Geo:       +{c-g:.3f} {"CONFIRMED" if c>g else "FAILED"}')
print(f'H3 Asym>Sym:       +{c-s:.3f} {"CONFIRMED" if c>s else "FAILED"}')


In [ ]:
# Figure 3
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size':9,'figure.dpi':300})
mo=['local_only','local_econ','geo_gat','symmetric_ecmp','cascrop']
dn=['Row 1:\nLocal Only','Row 2:\nLocal+Econ','Row 3:\nGeo GAT','Row 4:\nSym. ECMP','Row 5:\nCasCrop']
co=['#7f8c8d','#3498db','#e67e22','#9b59b6','#e74c3c']
ms=[df[df['model']==m]['test_auc_roc'].mean() for m in mo]
ss_=[df[df['model']==m]['test_auc_roc'].std() if len(df[df['model']==m])>1 else 0 for m in mo]
fig,ax=plt.subplots(figsize=(7,3.5))
bars=ax.bar(range(5),ms,0.6,yerr=ss_,capsize=4,color=co,edgecolor='k',linewidth=.5)
for b,v in zip(bars,ms):ax.text(b.get_x()+b.get_width()/2,b.get_height()+.008,f'{v:.3f}',ha='center',fontsize=7)
ax.set_xticks(range(5));ax.set_xticklabels(dn,fontsize=7)
ax.set_ylim(.7,1);ax.set_ylabel('AUC-ROC');ax.grid(axis='y',alpha=.3)
ax.set_title('Test Set Performance (2022-2024)',fontweight='bold')
plt.tight_layout();fig.savefig('paper/figures/fig3_ablation.pdf',dpi=300,bbox_inches='tight');plt.show()

In [ ]:
# LaTeX Table 2
names={'local_only':'Row 1: Local Only','local_econ':'Row 2: Local + Economic','geo_gat':'Row 3: Geographic GAT','symmetric_ecmp':'Row 4: Symmetric ECMP','cascrop':'Row 5: CasCrop$^\\dagger$'}
mets=['test_auc_roc','test_f1','test_auc_pr']
best={m:df.groupby('model')[m].mean().max() for m in mets}
rows=[]
for m in mo:
    d=df[df['model']==m]; cells=[names[m]]
    for met in mets:
        val=f'{d[met].mean():.3f} $\\pm$ {d[met].std():.3f}'
        if d[met].mean()==best[met]:val=f'\\textbf{{{val}}}'
        cells.append(val)
    cells.append(f"{d['n_params'].iloc[0]:,}")
    rows.append(' & '.join(cells)+' \\\\\\\\')
latex=f"""\\begin{{table*}}[t]
\\centering
\\caption{{Ablation results (test 2022-2024). Mean $\\pm$ std, {len(SEEDS)} seeds. $^\\dagger$Asymmetric ECMP.}}
\\label{{tab:ablation}}
\\begin{{tabular}}{{lcccc}}
\\toprule
Model & AUC-ROC & F1 & AUC-PR & Params \\\\\\\\
\\midrule
{chr(10).join(rows)}
\\bottomrule
\\end{{tabular}}
\\end{{table*}}"""
with open('paper/tables/table2_ablation.tex','w') as f:f.write(latex)
print(latex)

---
## Download

In [ ]:
backup()
!tar czf /content/cascrop_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/
try:
    from google.colab import files;files.download('/content/cascrop_results.tar.gz')
except:print('Download from Files or Drive')
print('ALL EXPERIMENTS COMPLETE')